# Pipeline ETL — Etapas E até H
## Consolidação Final de Bases Anonimizadas SUSEP

Este notebook implementa as etapas E até H do plano `PLANO_BASES_ANONIMIZADAS_SUSEP.md`:

| Etapa | Objetivo | Saída |
|-------|----------|-------|
| **E** | Merge em duas camadas (riscos R_ e sinistros S_) | `unified/parquet/riscos/`, `sinistros/` |
| **F** | Harmonização (tipos, sentinelas, periodo_analitico, premio_total_auto) | `unified/parquet/harmonizado_riscos/`, `harmonizado_sinistros/` |
| **G** | Controle de qualidade (contagens, domínios, nulos, stats) | `unified/quality_reports/run_*.md` |
| **H** | Exportação versionada + CSV resumo | `unified/parquet/base_susep_*_YYYYMMDD/` |

### Decisões técnicas

- **PyArrow nativo** (dataset + compute) — não Pandas — para suportar ~235M linhas sem OOM.
- **R_ e S_ em camadas separadas** — schemas e granularidades distintas; chaves de join por ramo.
- **Streaming em batches** em todas as etapas — nunca materializar o dataset inteiro em RAM.
- **Tipos**: Etapa D salvou tudo como `utf8`; a conversão para `float64` acontece na Etapa F.

## Seção 1 — Setup e Configuração

In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow.compute as pc
from pathlib import Path
from datetime import datetime
import json
import shutil
import csv as csv_mod
import logging
from typing import Dict, List, Optional

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ── Diretórios ──────────────────────────────────────────────────────────────
DATA_ROOT = Path("data/gov_br/susep/central-de-conteudos/dados-estatisticos/bases-anonimizadas")
UNIFIED_ROOT = Path("data/unified")

for subdir in ["parquet", "csv", "dictionaries", "quality_reports", "manifestos"]:
    (UNIFIED_ROOT / subdir).mkdir(parents=True, exist_ok=True)

# ── Mapeamento de ramos (idêntico ao ETL_A_D) ──────────────────────────────
RAMOS = {
    "bases_auto": {"ramo": "auto", "grupos": ["R_AUTO", "S_AUTO"]},
    "bases_comp": {"ramo": "comp", "grupos": ["R_COMP", "S_COMP"]},
    "bases_rural": {"ramo": "rural", "grupos": ["R_RURAL", "S_RURAL"]},
}

# ── Campos monetários/percentuais que devem ir para float64 (Etapa F) ──────
# Agrupados por grupo_arquivo para aplicar a conversão no fragmento correto.
CAMPOS_MONETARIOS = {
    "R_AUTO": [
        "val_franq", "perc_fator", "is_casco", "is_rcdmat", "is_rcdc", "is_rcdmor",
        "is_app_ma", "is_app_ipa", "is_app_dmh", "pre_casco", "pre_cas_co",
        "pre_rcdmat", "pre_rcdc", "pre_rcdmor", "pre_app_ma", "pre_app_ia",
        "pre_app_dm", "pre_outros", "perc_bonus", "perc_corr",
    ],
    "S_AUTO": ["indeniz", "val_salvad", "val_ress"],
    "R_COMP": ["val_franq", "imp_seg", "premio", "corretagem", "perc_desc"],
    "S_COMP": ["val_franq", "indeniz"],
    "R_RURAL": [
        "val_franq", "area_seg", "imp_seg", "premio", "premio_sub",
        "corretagem", "perc_carr", "perc_desc",
    ],
    "S_RURAL": ["indeniz", "desp_sin", "val_franq"],
}

# ── Campos de data (AAAAMMDD) — sentinela 00000000 → null ──────────────────
CAMPOS_DATA = {
    "R_AUTO": ["inicio_vig", "fim_vig", "data_nasc", "data_emis"],
    "S_AUTO": ["d_salvado", "d_ress", "d_avi", "d_liq", "d_ocorr", "d_nasc"],
    "R_COMP": ["inicio_vig", "fim_vig"],
    "S_COMP": ["d_aviso", "d_liq", "d_ocorr"],
    "R_RURAL": ["inicio_vig", "fim_vig"],
    "S_RURAL": ["d_aviso", "d_liq", "d_ocorr_in", "d_ocorr_fi"],
}

# ── Colunas que compõem premio_total_auto (soma para comparabilidade) ──────
PREMIO_COLS_AUTO = [
    "pre_casco", "pre_rcdmat", "pre_rcdc", "pre_rcdmor",
    "pre_app_ma", "pre_app_ia", "pre_app_dm", "pre_outros",
]

# ── Chaves de integração R↔S por ramo (documentadas nos manuais) ───────────
CHAVES_INTEGRACAO = {
    "auto": ["cod_apo", "cod_endosso", "item", "regiao"],
    "comp": ["cod_apo", "cod_endosso", "tipo", "classe", "item", "cobertura", "uf"],
    "rural": [
        "cod_apo", "cod_endosso", "cod_item", "cobertura",
        "cob_fundo", "cod_mod", "cultura", "munic", "uf", "id_bem",
    ],
}

# ── Domínios de código (para validação na Etapa G) ──────────────────────────
DOMINIOS = {
    ("cod_end", "auto"):  {"0", "1", "2", "3", "4"},
    ("cod_end", "comp"):  {"0", "1", "2", "3", "4"},
    ("cod_end", "rural"): {"0", "1", "2", "3", "4"},
    ("modalidade", "auto"): {"1", "2", "3", "4"},
    ("tipo", "comp"): {"1", "2", "3"},
    ("cob_fundo", "rural"): {"S", "N"},
    ("tipo_pes", "auto"): {"F", "J"},
    ("evento", "auto"): {str(i) for i in range(1, 9)},
    ("causa", "auto"): {"1", "2", "3", "4", "5", "6", "7", "9"},
    ("utilizacao", "auto"): {"0", "1", "2", "3"},
    ("sexo", "auto"): {"M", "F", "0"},
    ("tipo_franq", "auto"): {"1", "2", "3", "4", "9"},
    ("tipo_franq", "rural"): {"1", "2", "3", "4", "9"},
}

print(f"✓ Setup concluído — PyArrow {pa.__version__}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  UNIFIED_ROOT: {UNIFIED_ROOT}")

✓ Setup concluído — PyArrow 23.0.1
  DATA_ROOT:    data/gov_br/susep/central-de-conteudos/dados-estatisticos/bases-anonimizadas
  UNIFIED_ROOT: data/unified


## Seção 2 — Etapa E: Merge em Duas Camadas (Riscos e Sinistros)

Conforme seção E.3 do plano, R_ (riscos/prêmios) e S_ (sinistros) possuem granularidades
e semânticas distintas. O merge **não** mistura R_ com S_ em uma única tabela.

**Camada R (riscos/prêmios):** R_AUTO + R_COMP + R_RURAL → ~228,7M linhas
**Camada S (sinistros):** S_AUTO + S_COMP + S_RURAL → ~6,7M linhas

Colunas ausentes em um ramo ficam `null` nos demais (ex: `cultura` só existe em RURAL).
Nenhuma conversão de tipo nesta etapa — tudo permanece `utf8`.

In [2]:
def etapa_e_merge_camadas():
    """Etapa E: Merge em duas camadas separadas (riscos R_ e sinistros S_).

    Inventaria os Parquets padronizados, separa por prefixo R_/S_,
    unifica schemas por camada e grava os Parquets particionados por
    fragmento de origem. Colunas ausentes ficam null.
    """
    print("\n" + "=" * 70)
    print("ETAPA E — MERGE EM DUAS CAMADAS (RISCOS / SINISTROS)")
    print("=" * 70)

    # ── E.1  Inventariar todos os Parquets padronizados ─────────────────
    riscos_files = []    # (path, ramo, grupo)
    sinistros_files = [] # (path, ramo, grupo)
    resumo = []

    for base_folder, info in RAMOS.items():
        ramo = info["ramo"]
        std_path = DATA_ROOT / base_folder / "standardized"
        if not std_path.exists():
            print(f"  ⚠ Pasta não encontrada: {std_path}")
            continue

        for pf in sorted(std_path.glob("*.parquet")):
            meta = pq.read_metadata(pf)
            schema = pq.read_schema(pf)
            grupo = pf.stem.split("_")[0] + "_" + pf.stem.split("_")[1]  # R_AUTO, S_COMP, etc.

            entry = {
                "ramo": ramo, "grupo": grupo, "arquivo": pf.name,
                "linhas": meta.num_rows, "colunas": meta.num_columns,
                "tamanho_mb": round(pf.stat().st_size / 1e6, 1),
            }
            resumo.append(entry)

            if grupo.startswith("R_"):
                riscos_files.append((pf, ramo, grupo))
            else:
                sinistros_files.append((pf, ramo, grupo))

            print(f"  {grupo:8} {ramo:6} {pf.name:50} "
                  f"{meta.num_rows:>13,} linhas | {meta.num_columns:>3} cols")

    # ── E.2  Construir datasets lazy por camada ─────────────────────────
    resultados = {}
    for camada_nome, arquivos in [("riscos", riscos_files), ("sinistros", sinistros_files)]:
        if not arquivos:
            print(f"\n  ⚠ Nenhum arquivo para camada {camada_nome}")
            continue

        paths = [str(f) for f, _, _ in arquivos]
        schemas = [pq.read_schema(f) for f, _, _ in arquivos]
        unified = pa.unify_schemas(schemas)

        # Gravar Parquets por fragmento no diretório da camada
        out_dir = UNIFIED_ROOT / "parquet" / camada_nome
        if out_dir.exists():
            shutil.rmtree(out_dir)
        out_dir.mkdir(parents=True)

        total_linhas = 0
        for idx, (src_path, ramo, grupo) in enumerate(arquivos):
            out_file = out_dir / f"{grupo}_{ramo}_{idx:03d}.parquet"
            reader = pq.ParquetFile(src_path)

            writer = pq.ParquetWriter(out_file, unified, compression="snappy")
            for batch in reader.iter_batches():
                # Alinhar batch ao schema unificado: adicionar colunas faltantes como null
                columns = {}
                for field in unified:
                    if field.name in batch.schema.names:
                        col = batch.column(field.name)
                        if col.type != field.type:
                            col = pc.cast(col, field.type)
                        columns[field.name] = col
                    else:
                        columns[field.name] = pa.nulls(batch.num_rows, type=field.type)

                aligned = pa.record_batch(
                    [columns[f.name] for f in unified], schema=unified
                )
                writer.write_batch(aligned)
                total_linhas += batch.num_rows

            writer.close()
            size_mb = out_file.stat().st_size / 1e6
            print(f"  → {out_file.name:50} ({size_mb:>8.1f} MB)")

        # Abrir como dataset lazy
        dataset = ds.dataset(str(out_dir), format="parquet", schema=unified)
        resultados[camada_nome] = {
            "dataset": dataset,
            "schema": unified,
            "total_linhas": total_linhas,
            "dir": out_dir,
            "arquivo_info": [(ramo, grupo) for _, ramo, grupo in arquivos],
        }

        print(f"\n  ✓ Camada '{camada_nome}': {total_linhas:,} linhas | "
              f"{len(unified)} colunas | {camada_nome}/")

    # ── E.3  Salvar resumo do merge ─────────────────────────────────────
    resumo_file = UNIFIED_ROOT / "manifestos" / "resumo_merge.csv"
    with open(resumo_file, "w", newline="", encoding="utf-8") as f:
        w = csv_mod.DictWriter(f, fieldnames=["ramo", "grupo", "arquivo", "linhas", "colunas", "tamanho_mb"])
        w.writeheader()
        w.writerows(resumo)
    print(f"\n  ✓ Resumo salvo: {resumo_file}")

    return resultados


resultado_e = etapa_e_merge_camadas()


ETAPA E — MERGE EM DUAS CAMADAS (RISCOS / SINISTROS)
  R_AUTO   auto   R_AUTO_2020A_2020A_standardized.parquet               35,628,198 linhas |  52 cols
  R_AUTO   auto   R_AUTO_2020B_2020B_standardized.parquet               35,854,022 linhas |  52 cols
  S_AUTO   auto   S_AUTO_2020A_2020A_standardized.parquet                3,139,975 linhas |  30 cols
  S_AUTO   auto   S_AUTO_2020B_2020B_standardized.parquet                2,454,613 linhas |  30 cols
  R_COMP   comp   R_COMP_2022_2022_standardized.parquet                144,458,633 linhas |  21 cols
  S_COMP   comp   S_COMP_2022_2022_standardized.parquet                    911,357 linhas |  17 cols
  R_RURAL  rural  R_RURAL_2021_2021_standardized.parquet                12,773,353 linhas |  29 cols
  S_RURAL  rural  S_RURAL_2021_2021_standardized.parquet                   158,640 linhas |  24 cols
  → R_AUTO_auto_000.parquet                            (  1924.1 MB)
  → R_AUTO_auto_001.parquet                            (  1794.0 MB)


## Seção 3 — Etapa F: Harmonização de Variáveis

Conforme seções F.1–F.6 do plano:

1. **Conversão de tipos monetários** — `utf8` → `float64` com tratamento de sentinelas (`.` → `null`)
2. **Sentinelas de data** — `00000000` → `null`
3. **Chave temporal** — coluna `periodo_analitico` (`YYYYS` ou `YYYY`)
4. **Rastreabilidade** — coluna `fonte_susep_subpagina`
5. **Comparabilidade** — coluna `premio_total_auto` (soma dos `pre_*` em R_AUTO)
6. **Dicionários** — `variaveis_por_ramo.json` e `mapeamento_colunas.json`

In [6]:
def _safe_utf8_to_float(col: pa.Array) -> pa.Array:
    """Converte coluna utf8 → float64, tratando sentinelas (`.`, vazio) como null."""
    # Substituir "." e "" por null antes do cast
    is_dot = pc.equal(col, ".")
    is_empty = pc.equal(col, "")
    is_sentinel = pc.or_(is_dot, is_empty)
    cleaned = pc.if_else(is_sentinel, None, col)
    return pc.cast(cleaned, pa.float64(), safe=False)


def _nullify_date_sentinel(col: pa.Array) -> pa.Array:
    """Converte sentinela de data 00000000 → null (mantém utf8)."""
    is_zero = pc.equal(col, "00000000")
    return pc.if_else(is_zero, None, col)


# Conjunto global de todos os campos monetários (para que colunas herdadas
# via unify_schemas na Etapa E recebam float64 em TODOS os fragmentos).
_ALL_CAMPOS_MONETARIOS = set()
for _v in CAMPOS_MONETARIOS.values():
    _ALL_CAMPOS_MONETARIOS.update(_v)


def _build_harmonized_schema(src_schema: pa.Schema, grupo: str) -> pa.Schema:
    """Monta o schema de saída: converte tipos monetários para float64,
    adiciona colunas de harmonização."""
    fields = []
    for field in src_schema:
        if field.name in _ALL_CAMPOS_MONETARIOS:
            fields.append(pa.field(field.name, pa.float64()))
        else:
            fields.append(field)
    # Colunas adicionais de harmonização
    fields.append(pa.field("fonte_susep_subpagina", pa.utf8()))
    fields.append(pa.field("periodo_analitico", pa.utf8()))
    return pa.schema(fields)


def etapa_f_harmonizacao(resultado_e: Dict) -> Dict:
    """Etapa F: Harmonização — converte tipos, trata sentinelas, adiciona colunas derivadas.

    Processa cada camada (riscos/sinistros) em streaming por fragmento Parquet.
    Para R_AUTO, calcula premio_total_auto como soma dos pre_*.
    """
    print("\n" + "=" * 70)
    print("ETAPA F — HARMONIZAÇÃO DE VARIÁVEIS")
    print("=" * 70)

    if resultado_e is None:
        print("  ⚠ Resultado da Etapa E não disponível!")
        return None

    resultados_f = {}
    dicionario_variaveis = {}
    colunas_por_ramo = {}

    for camada_nome, camada_info in resultado_e.items():
        src_dir = camada_info["dir"]
        out_dir = UNIFIED_ROOT / "parquet" / f"harmonizado_{camada_nome}"
        if out_dir.exists():
            shutil.rmtree(out_dir)
        out_dir.mkdir(parents=True)

        print(f"\n  Processando camada: {camada_nome}")

        # Coletar schemas de saída por fragmento para unificação posterior
        out_schemas = []
        total_linhas = 0
        contadores_ramo = {}

        for src_file in sorted(src_dir.glob("*.parquet")):
            # Extrair grupo e ramo do nome: R_AUTO_auto_000.parquet
            parts = src_file.stem.split("_")
            grupo = parts[0] + "_" + parts[1]  # R_AUTO, S_COMP, etc.
            ramo = parts[2]                      # auto, comp, rural

            reader = pq.ParquetFile(src_file)
            src_schema = reader.schema_arrow

            # Campos monetários (global para consistência de tipos) e de data
            float_fields = _ALL_CAMPOS_MONETARIOS & set(src_schema.names)
            date_fields = set(CAMPOS_DATA.get(grupo, []))

            # Schema de saída para este fragmento
            out_schema = _build_harmonized_schema(src_schema, grupo)

            # Se R_AUTO, adicionar premio_total_auto
            if grupo == "R_AUTO":
                out_schema = out_schema.append(pa.field("premio_total_auto", pa.float64()))

            out_schemas.append(out_schema)
            out_file = out_dir / f"{grupo}_{ramo}_{src_file.stem.split('_')[-1]}.parquet"
            writer = pq.ParquetWriter(out_file, out_schema, compression="snappy")

            frag_linhas = 0
            for batch in reader.iter_batches():
                n = batch.num_rows
                frag_linhas += n
                total_linhas += n
                contadores_ramo[ramo] = contadores_ramo.get(ramo, 0) + n

                new_columns = {}
                for field in src_schema:
                    col = batch.column(field.name)
                    if field.name in float_fields:
                        col = _safe_utf8_to_float(col)
                    elif field.name in date_fields:
                        col = _nullify_date_sentinel(col)
                    new_columns[field.name] = col

                # Coluna fonte_susep_subpagina (derivada do ramo)
                new_columns["fonte_susep_subpagina"] = pa.array([ramo] * n, type=pa.utf8())

                # Coluna periodo_analitico: YYYYS se semestre presente, senão YYYY
                col_ano = pc.cast(batch.column("ano"), pa.utf8())
                col_semestre = batch.column("semestre")
                if col_semestre.type != pa.utf8():
                    col_semestre = pc.cast(col_semestre, pa.utf8())
                sem_valido = pc.is_valid(col_semestre)
                com_sem = pc.binary_join_element_wise(col_ano, col_semestre, "")
                new_columns["periodo_analitico"] = pc.if_else(sem_valido, com_sem, col_ano)

                # premio_total_auto (soma dos pre_*)
                if grupo == "R_AUTO":
                    total_premio = pa.nulls(n, type=pa.float64())
                    for pcol in PREMIO_COLS_AUTO:
                        if pcol in new_columns:
                            vals = new_columns[pcol]
                        else:
                            vals = pa.nulls(n, type=pa.float64())
                        # pc.add trata null como null propagado; usamos fill_null(0) para soma
                        total_premio = pc.add(
                            pc.fill_null(total_premio, 0.0),
                            pc.fill_null(vals, 0.0),
                        )
                    new_columns["premio_total_auto"] = total_premio

                out_batch = pa.record_batch(
                    [new_columns[f.name] for f in out_schema],
                    schema=out_schema,
                )
                writer.write_batch(out_batch)

            writer.close()
            size_mb = out_file.stat().st_size / 1e6
            print(f"    ✓ {out_file.name:45} {frag_linhas:>13,} linhas | {size_mb:>8.1f} MB")

            # Registrar colunas por ramo
            if ramo not in colunas_por_ramo:
                colunas_por_ramo[ramo] = {}
            colunas_por_ramo[ramo][camada_nome] = {
                "colunas": out_schema.names,
                "tipos": [str(out_schema.field(c).type) for c in out_schema.names],
            }

        # Reabrir como dataset com schema unificado
        all_schemas = [pq.read_schema(f) for f in sorted(out_dir.glob("*.parquet"))]
        unified = pa.unify_schemas(all_schemas)
        dataset = ds.dataset(str(out_dir), format="parquet", schema=unified)

        resultados_f[camada_nome] = {
            "dataset": dataset,
            "schema": unified,
            "total_linhas": total_linhas,
            "dir": out_dir,
            "contadores_ramo": contadores_ramo,
        }

        total_size = sum(f.stat().st_size for f in out_dir.glob("*.parquet"))
        print(f"\n  ✓ Camada '{camada_nome}' harmonizada: {total_linhas:,} linhas | "
              f"{len(unified)} colunas | {total_size / 1e6:.1f} MB")

    # ── Gerar dicionário de variáveis ───────────────────────────────────
    for ramo in ["auto", "comp", "rural"]:
        info_r = colunas_por_ramo.get(ramo, {}).get("riscos", {})
        info_s = colunas_por_ramo.get(ramo, {}).get("sinistros", {})
        r_grupo = f"R_{ramo.upper()}"
        s_grupo = f"S_{ramo.upper()}"
        dicionario_variaveis[ramo] = {
            "periodicidade": "semestral" if ramo == "auto" else "anual",
            "chaves_integracao_R_S": CHAVES_INTEGRACAO.get(ramo, []),
            "num_registros_R": resultados_f.get("riscos", {}).get("contadores_ramo", {}).get(ramo, 0),
            "num_registros_S": resultados_f.get("sinistros", {}).get("contadores_ramo", {}).get(ramo, 0),
            "colunas_monetarias_R": CAMPOS_MONETARIOS.get(r_grupo, []),
            "colunas_monetarias_S": CAMPOS_MONETARIOS.get(s_grupo, []),
            "colunas_data_R": CAMPOS_DATA.get(r_grupo, []),
            "colunas_data_S": CAMPOS_DATA.get(s_grupo, []),
            "schema_riscos": info_r.get("colunas", []),
            "schema_sinistros": info_s.get("colunas", []),
        }

    dict_file = UNIFIED_ROOT / "dictionaries" / "variaveis_por_ramo.json"
    with open(dict_file, "w", encoding="utf-8") as f:
        json.dump(dicionario_variaveis, f, indent=2, ensure_ascii=False)
    print(f"\n  ✓ Dicionário de variáveis: {dict_file}")

    # ── Gerar mapeamento de colunas ─────────────────────────────────────
    colunas_comuns = [
        "ramo", "grupo_arquivo", "ano", "semestre", "arquivo_origem",
        "fonte_susep_subpagina", "periodo_analitico",
    ]
    colunas_especificas = {}
    for ramo, camadas in colunas_por_ramo.items():
        all_cols = set()
        for info in camadas.values():
            all_cols.update(info.get("colunas", []))
        colunas_especificas[ramo] = sorted(c for c in all_cols if c not in colunas_comuns)

    mapa = {"colunas_comuns": colunas_comuns, "colunas_especificas_por_ramo": colunas_especificas}
    mapa_file = UNIFIED_ROOT / "dictionaries" / "mapeamento_colunas.json"
    with open(mapa_file, "w", encoding="utf-8") as f:
        json.dump(mapa, f, indent=2, ensure_ascii=False)

    print(f"  ✓ Mapeamento de colunas: {mapa_file}")

    return resultados_f


resultado_f = etapa_f_harmonizacao(resultado_e)


ETAPA F — HARMONIZAÇÃO DE VARIÁVEIS

  Processando camada: riscos
    ✓ R_AUTO_auto_000.parquet                          35,628,198 linhas |   1977.7 MB
    ✓ R_AUTO_auto_001.parquet                          35,854,022 linhas |   1845.5 MB
    ✓ R_COMP_comp_002.parquet                         144,458,633 linhas |   2014.6 MB
    ✓ R_RURAL_rural_003.parquet                        12,773,353 linhas |    341.7 MB

  ✓ Camada 'riscos' harmonizada: 228,714,206 linhas | 72 colunas | 6179.5 MB

  Processando camada: sinistros
    ✓ S_AUTO_auto_000.parquet                           3,139,975 linhas |     93.7 MB
    ✓ S_AUTO_auto_001.parquet                           2,454,613 linhas |     75.2 MB
    ✓ S_COMP_comp_002.parquet                             911,357 linhas |     13.8 MB
    ✓ S_RURAL_rural_003.parquet                           158,640 linhas |      5.2 MB

  ✓ Camada 'sinistros' harmonizada: 6,664,585 linhas | 47 colunas | 187.8 MB

  ✓ Dicionário de variáveis: data/unified/dicti

## Seção 4 — Etapa G: Controle de Qualidade

Conforme seções G.1–G.6 do plano, as validações são executadas em **single-pass streaming**
(nunca materializa o dataset em RAM):

1. Contagem de linhas antes/depois (Etapa D vs. Etapa F)
2. Domínios de código (manuais SUSEP)
3. Análise de nulos (estrutural vs. faltante)
4. Estatísticas monetárias (min, max, mean, negativos)
5. Relatório gerado em `quality_reports/run_YYYYMMDD_HHMM.md`

In [8]:
def etapa_g_controle_qualidade(resultado_f: Dict) -> Dict:
    """Etapa G: Controle de qualidade via scanning em batches (single-pass).

    Verifica:
    - G.1: Completude do pipeline
    - G.2: Contagem de linhas antes/depois
    - G.3: Domínios de código
    - G.4: Análise de nulos (estrutural vs. faltante)
    - G.5: Estatísticas monetárias
    """
    print("\n" + "=" * 70)
    print("ETAPA G — CONTROLE DE QUALIDADE")
    print("=" * 70)

    if resultado_f is None:
        print("  ⚠ Resultado da Etapa F não disponível!")
        return None

    qualidade = {"alertas": []}
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    # ── G.1  Validação de completude ────────────────────────────────────
    manifest_file = UNIFIED_ROOT / "manifestos" / "links_checksums_manifest.csv"
    if manifest_file.exists():
        import pyarrow.csv as pa_csv
        tbl_m = pa_csv.read_csv(str(manifest_file))
        qualidade["zips_esperados"] = tbl_m.num_rows
        print(f"\n  ✓ G.1  ZIPs no manifesto: {tbl_m.num_rows}")
    else:
        qualidade["zips_esperados"] = "N/A"
        print("\n  ⚠ G.1  Manifesto não encontrado")

    # ── G.2  Contagem antes/depois ──────────────────────────────────────
    print("\n  ✓ G.2  Contagem de linhas por camada:")
    # Ler contagens da Etapa D (Parquets padronizados)
    contagens_d = {}
    for base_folder, info in RAMOS.items():
        ramo = info["ramo"]
        std_path = DATA_ROOT / base_folder / "standardized"
        if not std_path.exists():
            continue
        for pf in sorted(std_path.glob("*.parquet")):
            grupo = pf.stem.split("_")[0] + "_" + pf.stem.split("_")[1]
            meta = pq.read_metadata(pf)
            chave = (ramo, grupo)
            contagens_d[chave] = contagens_d.get(chave, 0) + meta.num_rows

    contagens_f = {}
    for camada_nome, camada_info in resultado_f.items():
        for ramo, count in camada_info["contadores_ramo"].items():
            prefixo = "R_" if camada_nome == "riscos" else "S_"
            grupo = prefixo + ramo.upper()
            contagens_f[(ramo, grupo)] = contagens_f.get((ramo, grupo), 0) + count

    qualidade["contagens"] = []
    for chave in sorted(set(contagens_d) | set(contagens_f)):
        d_count = contagens_d.get(chave, 0)
        f_count = contagens_f.get(chave, 0)
        ok = d_count == f_count
        status = "✓" if ok else "✗"
        if not ok:
            qualidade["alertas"].append(f"Contagem divergente: {chave} D={d_count:,} F={f_count:,}")
        qualidade["contagens"].append({
            "ramo": chave[0], "grupo": chave[1],
            "etapa_d": d_count, "etapa_f": f_count, "ok": ok,
        })
        print(f"    {status} {chave[1]:8} {chave[0]:6} | D={d_count:>13,} | F={f_count:>13,}")

    # ── G.3  Domínios de código + G.4  Nulos + G.5  Stats monetários ───
    print("\n  ✓ G.3–G.5  Scanning em batches por camada...")
    qualidade["nulos"] = {}
    qualidade["stats_monetarios"] = {}
    qualidade["dominios_invalidos"] = {}

    for camada_nome, camada_info in resultado_f.items():
        dataset = camada_info["dataset"]
        schema = camada_info["schema"]
        total = camada_info["total_linhas"]

        nulos = {col: 0 for col in schema.names}
        negativos = {}
        soma = {}
        minimos = {}
        maximos = {}
        contagem_validos = {}
        dominios_encontrados = {}

        for batch in dataset.to_batches():
            # Extrair ramo do batch para validação de domínios
            ramo_arr = batch.column("ramo")
            ramos_no_batch = pc.unique(ramo_arr).to_pylist()

            for col_name in schema.names:
                col = batch.column(col_name)

                # Nulos
                n_null = pc.sum(pc.is_null(col)).as_py() or 0
                nulos[col_name] += n_null

                # Stats monetários (float64)
                if schema.field(col_name).type == pa.float64():
                    valid = pc.filter(col, pc.is_valid(col))
                    if len(valid) > 0:
                        contagem_validos[col_name] = contagem_validos.get(col_name, 0) + len(valid)
                        batch_min = pc.min(valid).as_py()
                        batch_max = pc.max(valid).as_py()
                        batch_sum = pc.sum(valid).as_py() or 0.0
                        batch_neg = pc.sum(pc.less(valid, 0)).as_py() or 0

                        if col_name not in minimos or (batch_min is not None and batch_min < minimos[col_name]):
                            minimos[col_name] = batch_min
                        if col_name not in maximos or (batch_max is not None and batch_max > maximos[col_name]):
                            maximos[col_name] = batch_max
                        soma[col_name] = soma.get(col_name, 0.0) + batch_sum
                        negativos[col_name] = negativos.get(col_name, 0) + batch_neg

                # Domínios de código
                for ramo in ramos_no_batch:
                    dom_key = (col_name, ramo)
                    if dom_key in DOMINIOS:
                        # Filtrar batch pelo ramo
                        mask = pc.equal(ramo_arr, ramo)
                        col_ramo = pc.filter(col, mask)
                        vals = set(pc.unique(pc.filter(col_ramo, pc.is_valid(col_ramo))).to_pylist())
                        if dom_key not in dominios_encontrados:
                            dominios_encontrados[dom_key] = set()
                        dominios_encontrados[dom_key] |= vals

        # Registrar nulos por camada
        qualidade["nulos"][camada_nome] = {
            col: {"count": count, "pct": round(count / total * 100, 2) if total > 0 else 0}
            for col, count in nulos.items() if count > 0
        }

        # Registrar stats monetários
        for col_name in contagem_validos:
            cnt = contagem_validos[col_name]
            qualidade["stats_monetarios"][f"{camada_nome}.{col_name}"] = {
                "min": minimos.get(col_name),
                "max": maximos.get(col_name),
                "mean": round(soma.get(col_name, 0) / cnt, 2) if cnt > 0 else None,
                "count_valid": cnt,
                "count_null": nulos.get(col_name, 0),
                "count_negative": negativos.get(col_name, 0),
            }

        # Validar domínios
        for dom_key, vals_encontrados in dominios_encontrados.items():
            dom_esperado = DOMINIOS[dom_key]
            invalidos = vals_encontrados - dom_esperado
            if invalidos:
                alerta = f"Domínio {dom_key[0]} ({dom_key[1]}): valores inesperados: {sorted(invalidos)[:10]}"
                qualidade["alertas"].append(alerta)
                qualidade["dominios_invalidos"][f"{dom_key[0]}_{dom_key[1]}"] = sorted(invalidos)[:20]

        print(f"    ✓ {camada_nome}: {total:,} linhas escaneadas")

    # ── G.6  Gerar relatório Markdown ───────────────────────────────────
    report_file = UNIFIED_ROOT / "quality_reports" / f"run_{timestamp}.md"

    ramos_processados = sorted(set(
        c["ramo"] for c in qualidade.get("contagens", [])
    ))

    total_geral = sum(
        camada_info["total_linhas"] for camada_info in resultado_f.values()
    )

    report = [
        f"# Relatório de Qualidade — Pipeline ETL SUSEP",
        f"## Data/Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"",
        f"## Resumo Executivo",
        f"- **Total de registros**: {total_geral:,}",
        f"- **Ramos**: {', '.join(ramos_processados)}",
        f"- **Engine**: PyArrow {pa.__version__} (streaming/batches)",
        f"- **Alertas**: {len(qualidade['alertas'])}",
        f"",
        f"## G.1 Completude",
        f"- ZIPs no manifesto: {qualidade.get('zips_esperados', 'N/A')}",
        f"",
        f"## G.2 Contagem de linhas (Etapa D → Etapa F)",
        f"",
        f"| Ramo | Grupo | Etapa D | Etapa F | Status |",
        f"|------|-------|---------|---------|--------|",
    ]
    for c in qualidade.get("contagens", []):
        status = "✓" if c["ok"] else "✗ DIVERGENTE"
        report.append(f"| {c['ramo']} | {c['grupo']} | {c['etapa_d']:,} | {c['etapa_f']:,} | {status} |")

    report.append(f"\n## G.3 Domínios de código")
    if qualidade["dominios_invalidos"]:
        for key, vals in qualidade["dominios_invalidos"].items():
            report.append(f"- **{key}**: valores inesperados: `{vals}`")
    else:
        report.append("- ✓ Todos os domínios dentro do esperado")

    report.append(f"\n## G.4 Análise de nulos (top 20 por camada)")
    for camada_nome, nulos_dict in qualidade.get("nulos", {}).items():
        report.append(f"\n### {camada_nome}")
        report.append(f"| Coluna | Nulos | % |")
        report.append(f"|--------|-------|---|")
        top = sorted(nulos_dict.items(), key=lambda x: x[1]["count"], reverse=True)[:20]
        for col, info in top:
            report.append(f"| {col} | {info['count']:,} | {info['pct']:.2f}% |")

    report.append(f"\n## G.5 Estatísticas monetárias")
    report.append(f"| Campo | Min | Max | Média | Válidos | Nulos | Negativos |")
    report.append(f"|-------|-----|-----|-------|---------|-------|-----------|")
    for key, stats in sorted(qualidade.get("stats_monetarios", {}).items()):
        report.append(
            f"| {key} | {stats['min']} | {stats['max']} | {stats['mean']} | "
            f"{stats['count_valid']:,} | {stats['count_null']:,} | {stats['count_negative']:,} |"
        )

    if qualidade["alertas"]:
        report.append(f"\n## Alertas ({len(qualidade['alertas'])})")
        for a in qualidade["alertas"]:
            report.append(f"- ⚠ {a}")

    report.append(f"\n---\n*Gerado automaticamente pelo Pipeline ETL SUSEP (PyArrow)*")

    with open(report_file, "w", encoding="utf-8") as f:
        f.write("\n".join(report))
    print(f"\n  ✓ Relatório de qualidade: {report_file}")

    qualidade["report_file"] = str(report_file)
    qualidade["timestamp"] = timestamp
    return qualidade


qualidade = etapa_g_controle_qualidade(resultado_f)


ETAPA G — CONTROLE DE QUALIDADE

  ✓ G.1  ZIPs no manifesto: 8

  ✓ G.2  Contagem de linhas por camada:
    ✓ R_AUTO   auto   | D=   71,482,220 | F=   71,482,220
    ✓ S_AUTO   auto   | D=    5,594,588 | F=    5,594,588
    ✓ R_COMP   comp   | D=  144,458,633 | F=  144,458,633
    ✓ S_COMP   comp   | D=      911,357 | F=      911,357
    ✓ R_RURAL  rural  | D=   12,773,353 | F=   12,773,353
    ✓ S_RURAL  rural  | D=      158,640 | F=      158,640

  ✓ G.3–G.5  Scanning em batches por camada...
    ✓ riscos: 228,714,206 linhas escaneadas
    ✓ sinistros: 6,664,585 linhas escaneadas

  ✓ Relatório de qualidade: data/unified/quality_reports/run_20260331_1414.md


## Seção 5 — Etapa H: Exportação Final e Versionamento

Conforme seções H.1–H.4 do plano:

1. **Cópia versionada** — `unified/parquet/base_susep_{camada}_YYYYMMDD_HHMM/`
2. **Link "latest"** — `unified/parquet/base_susep_{camada}_latest/`
3. **CSV resumo** — `unified/csv/base_susep_anonimizada_resumo_YYYYMMDD_HHMM.csv`

In [9]:
def etapa_h_exportacao_final(resultado_f: Dict, qualidade: Dict) -> Dict:
    """Etapa H: Exportação Final — versionamento, latest, CSV resumo."""
    print("\n" + "=" * 70)
    print("ETAPA H — EXPORTAÇÃO FINAL E VERSIONAMENTO")
    print("=" * 70)

    if resultado_f is None:
        print("  ⚠ Resultado da Etapa F não disponível!")
        return None

    timestamp = qualidade.get("timestamp", datetime.now().strftime("%Y%m%d_%H%M"))
    exportacoes = {}

    # ── H.1  Cópia versionada + latest por camada ──────────────────────
    for camada_nome, camada_info in resultado_f.items():
        src_dir = camada_info["dir"]

        # Versionado
        versioned = UNIFIED_ROOT / "parquet" / f"base_susep_{camada_nome}_{timestamp}"
        if versioned.exists():
            shutil.rmtree(versioned)
        shutil.copytree(src_dir, versioned)
        size_mb = sum(f.stat().st_size for f in versioned.glob("*.parquet")) / 1e6
        exportacoes[f"parquet_{camada_nome}_versionado"] = str(versioned)
        print(f"\n  ✓ Versionado: {versioned.name}/ ({size_mb:.1f} MB)")

        # Latest
        latest = UNIFIED_ROOT / "parquet" / f"base_susep_{camada_nome}_latest"
        if latest.exists():
            shutil.rmtree(latest)
        shutil.copytree(src_dir, latest)
        exportacoes[f"parquet_{camada_nome}_latest"] = str(latest)
        print(f"  ✓ Latest:     {latest.name}/")

    # ── H.2  CSV resumo ────────────────────────────────────────────────
    csv_file = UNIFIED_ROOT / "csv" / f"base_susep_anonimizada_resumo_{timestamp}.csv"

    contagem = {}
    for camada_nome, camada_info in resultado_f.items():
        dataset = camada_info["dataset"]
        for batch in dataset.to_batches(columns=["ramo", "periodo_analitico", "grupo_arquivo"]):
            ramos = batch.column("ramo").to_pylist()
            periodos = batch.column("periodo_analitico").to_pylist()
            grupos = batch.column("grupo_arquivo").to_pylist()
            for r, p, g in zip(ramos, periodos, grupos):
                chave = (r, p, g, camada_nome)
                contagem[chave] = contagem.get(chave, 0) + 1

    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        w = csv_mod.writer(f)
        w.writerow(["ramo", "periodo_analitico", "grupo_arquivo", "camada", "num_registros"])
        for (r, p, g, c), count in sorted(contagem.items()):
            w.writerow([r, p, g, c, count])

    exportacoes["csv_resumo"] = str(csv_file)
    print(f"\n  ✓ CSV resumo: {csv_file.name} ({len(contagem)} combinações)")

    # ── H.3  Listar todos os metadados gerados ─────────────────────────
    print(f"\n  ✓ Metadados e dicionários:")
    for label, path in [
        ("Manifesto", UNIFIED_ROOT / "manifestos" / "links_checksums_manifest.csv"),
        ("Resumo merge", UNIFIED_ROOT / "manifestos" / "resumo_merge.csv"),
        ("Dicionário variáveis", UNIFIED_ROOT / "dictionaries" / "variaveis_por_ramo.json"),
        ("Mapeamento colunas", UNIFIED_ROOT / "dictionaries" / "mapeamento_colunas.json"),
        ("Relatório qualidade", qualidade.get("report_file", "")),
    ]:
        existe = Path(path).exists() if path else False
        status = "✓" if existe else "—"
        print(f"    {status} {label}: {Path(path).name if path else 'N/A'}")

    return exportacoes


exportacoes = etapa_h_exportacao_final(resultado_f, qualidade)


ETAPA H — EXPORTAÇÃO FINAL E VERSIONAMENTO

  ✓ Versionado: base_susep_riscos_20260331_1414/ (6179.5 MB)
  ✓ Latest:     base_susep_riscos_latest/

  ✓ Versionado: base_susep_sinistros_20260331_1414/ (187.8 MB)
  ✓ Latest:     base_susep_sinistros_latest/

  ✓ CSV resumo: base_susep_anonimizada_resumo_20260331_1414.csv (8 combinações)

  ✓ Metadados e dicionários:
    ✓ Manifesto: links_checksums_manifest.csv
    ✓ Resumo merge: resumo_merge.csv
    ✓ Dicionário variáveis: variaveis_por_ramo.json
    ✓ Mapeamento colunas: mapeamento_colunas.json
    ✓ Relatório qualidade: run_20260331_1414.md


## Seção 6 — Resumo Final

In [10]:
print("\n" + "=" * 70)
print("RESUMO FINAL — PIPELINE ETAPAS E–H")
print("=" * 70)

if resultado_f is not None:
    for camada_nome, info in resultado_f.items():
        total = info["total_linhas"]
        ncols = len(info["schema"])
        print(f"\n  Camada '{camada_nome}':")
        print(f"    Total: {total:,} linhas | {ncols} colunas")
        for ramo, count in sorted(info["contadores_ramo"].items()):
            pct = count / total * 100 if total > 0 else 0
            print(f"    {ramo:6}: {count:>13,} ({pct:>5.1f}%)")

if qualidade is not None:
    n_alertas = len(qualidade.get("alertas", []))
    print(f"\n  Alertas de qualidade: {n_alertas}")
    if n_alertas > 0:
        for a in qualidade["alertas"][:5]:
            print(f"    ⚠ {a}")
        if n_alertas > 5:
            print(f"    ... e mais {n_alertas - 5} alertas (ver relatório)")

if exportacoes is not None:
    print(f"\n  Arquivos exportados:")
    for chave, caminho in exportacoes.items():
        p = Path(caminho)
        if p.is_dir():
            size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6
            print(f"    {chave:35}: {p.name}/ ({size:.1f} MB)")
        elif p.exists():
            print(f"    {chave:35}: {p.name} ({p.stat().st_size / 1e6:.1f} MB)")

print(f"\n  Diretórios utilizados:")
for d in ["parquet", "csv", "dictionaries", "quality_reports", "manifestos"]:
    print(f"    data/unified/{d}/")

print("\n" + "=" * 70)
print("✓ PIPELINE ETAPAS E–H CONCLUÍDO")
print("=" * 70)


RESUMO FINAL — PIPELINE ETAPAS E–H

  Camada 'riscos':
    Total: 228,714,206 linhas | 72 colunas
    auto  :    71,482,220 ( 31.3%)
    comp  :   144,458,633 ( 63.2%)
    rural :    12,773,353 (  5.6%)

  Camada 'sinistros':
    Total: 6,664,585 linhas | 47 colunas
    auto  :     5,594,588 ( 83.9%)
    comp  :       911,357 ( 13.7%)
    rural :       158,640 (  2.4%)

  Alertas de qualidade: 6
    ⚠ Domínio cod_end (comp): valores inesperados: ['5', '7']
    ⚠ Domínio tipo (comp): valores inesperados: ['0']
    ⚠ Domínio tipo_franq (rural): valores inesperados: ['0', '7']
    ⚠ Domínio modalidade (auto): valores inesperados: ['0']
    ⚠ Domínio evento (auto): valores inesperados: ['9']
    ... e mais 1 alertas (ver relatório)

  Arquivos exportados:
    parquet_riscos_versionado          : base_susep_riscos_20260331_1414/ (6179.5 MB)
    parquet_riscos_latest              : base_susep_riscos_latest/ (6179.5 MB)
    parquet_sinistros_versionado       : base_susep_sinistros_20260331_1

## Pipeline Completo

### Etapas executadas:
1. **E — Merge em duas camadas**: R_ (riscos/prêmios ~228M linhas) e S_ (sinistros ~6.7M linhas) separados, com `pa.unify_schemas()` para resolver schemas heterogêneos.
2. **F — Harmonização**: Conversão monetários→float64, sentinelas→null, `periodo_analitico`, `fonte_susep_subpagina`, `premio_total_auto` (R_AUTO).
3. **G — Qualidade**: Contagens D→F, domínios SUSEP, nulos estruturais vs. faltantes, stats monetários − relatório em Markdown.
4. **H — Exportação**: Versionado + latest + CSV resumo.

### Decisão: por que duas camadas (não uma tabela única)?
- **R_** e **S_** têm granularidades distintas (R = apólice/endosso/item/cobertura, S = sinistro individual)
- **Chaves de join** R↔S variam por ramo (ver `CHAVES_INTEGRACAO` no setup)
- **Schemas**: R_AUTO tem 47 cols dados, R_COMP 16, R_RURAL 24 — tabela plana teria ~70% null
- **Performance**: `pyarrow.dataset` permite leitura lazy com filtro pushdown por ramo/período

### Arquivos gerados:
- `data/unified/parquet/harmonizado_riscos/` — Parquets particionados (riscos)
- `data/unified/parquet/harmonizado_sinistros/` — Parquets particionados (sinistros)
- `data/unified/parquet/base_susep_{riscos,sinistros}_YYYYMMDD_HHMM/` — Versionados
- `data/unified/parquet/base_susep_{riscos,sinistros}_latest/` — Latest
- `data/unified/csv/base_susep_anonimizada_resumo_*.csv` — Resumo
- `data/unified/dictionaries/variaveis_por_ramo.json` — Dicionário
- `data/unified/dictionaries/mapeamento_colunas.json` — Mapeamento
- `data/unified/quality_reports/run_*.md` — Relatório de qualidade